# Phân cụm giao dịch IBFT bằng Autoencoder + K-Means

Autoencoder nén 11 đặc trưng xuống 4 chiều, sau đó K-Means phân cụm trên latent.
Một điểm dữ liệu là một giao dịch; số tài khoản không đi vào mô hình.

**Quy trình:** fit trên train → chọn K và seed bằng validation → báo cáo cuối trên test.
Ba chỉ số phân cụm luôn tính trên **11 đặc trưng `x_*` gốc** để so sánh với K-Means và GMM.

**Cách chạy:** chọn kernel có PyTorch, rồi *Restart Kernel → Run All*.
Đọc `output/prepared/`, ghi kết quả vào `output/autoencoder_kmeans/`.

## 1. Cấu hình

Các tham số cần thay đổi nằm trong ô này. `RUN_ID` là tên lần chạy trong cấu hình;
kết quả luôn ghi vào `output/autoencoder_kmeans/`.
Mạng chạy trên CPU; một luồng PyTorch phù hợp với mạng nhỏ này.

In [ ]:
RUN_ID = "run_001"
K_GRID = list(range(3, 11))
SEEDS = [42, 43, 44, 45, 46]
SILHOUETTE_TRAIN_SIZE = 10_000
PCA_SAMPLE_SIZE = 5_000
EVAL_SEED = 42
TORCH_THREADS = 1

AE_PARAMS = dict(
    architecture=[11, 16, 8, 4, 8, 16, 11],
    lr=1e-3, batch_size=256, max_epochs=100, patience=10,
    inference_batch_size=4096,
)
KMEANS_PARAMS = dict(
    init="k-means++", n_init=10, max_iter=300, tol=1e-4, algorithm="lloyd",
)


## 2. Thư viện và đường dẫn

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def find_project_root():
    """Đi ngược lên các thư mục cha để tìm thư mục dự án (nơi có output/prepared)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "output" / "prepared" / "model_input.parquet").is_file():
            return folder
    raise FileNotFoundError("Chưa có model_input.parquet; hãy chạy notebook 01 trước.")


PROJECT_ROOT = find_project_root()
PREPARED_DIR = PROJECT_ROOT / "output" / "prepared"
RUN_DIR = PROJECT_ROOT / "output"
METHOD_DIR = RUN_DIR / "autoencoder_kmeans"
FIGURES_DIR = METHOD_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def save_figure(fig, filename):
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Dữ liệu:", PREPARED_DIR)
print("Kết quả:", METHOD_DIR)
# Giữ kết quả lặp lại được trên cùng môi trường CPU.
torch.set_num_threads(TORCH_THREADS)
torch.use_deterministic_algorithms(True)


## 3. Đọc dữ liệu dùng chung

Giữ nguyên đặc trưng và cách chia tập từ notebook 01. Không điền thiếu, chuẩn hóa
hay tính lại lịch sử ở đây. Các `assert` chỉ kiểm tra những điều cốt lõi: đủ 11 cột,
không thiếu giá trị, và train → validation → test nối tiếp nhau theo thời gian.

In [ ]:
RAW_FEATURES = [
    "log_amount", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "log_sender_recency_days", "log_sender_tx_count_7d", "log_pair_tx_count_7d",
    "receiver_share_7d", "is_first_sender_tx", "is_new_receiver",
]
FEATURE_COLUMNS = [f"x_{name}" for name in RAW_FEATURES]
SPLITS = ["train", "validation", "test"]

model_input = pd.read_parquet(PREPARED_DIR / "model_input.parquet").reset_index(drop=True)

# Kiểm tra cốt lõi
assert all(col in model_input.columns for col in FEATURE_COLUMNS), "Thiếu cột đặc trưng x_*"
assert np.isfinite(model_input[FEATURE_COLUMNS].to_numpy()).all(), "Đặc trưng có giá trị thiếu hoặc vô hạn"
assert model_input["transaction_id"].is_unique, "transaction_id bị trùng"
assert set(model_input["split"]) == set(SPLITS), "split phải gồm train/validation/test"

split_summary = (
    model_input.groupby("split")
    .agg(row_count=("transaction_id", "size"),
         start_time=("transaction_datetime", "min"),
         end_time=("transaction_datetime", "max"))
    .reindex(SPLITS)
)
assert split_summary.loc["train", "end_time"] < split_summary.loc["validation", "start_time"]
assert split_summary.loc["validation", "end_time"] < split_summary.loc["test", "start_time"]

# Ma trận đặc trưng của từng tập
split_masks = {s: model_input["split"].eq(s).to_numpy() for s in SPLITS}
X = {s: model_input.loc[split_masks[s], FEATURE_COLUMNS].to_numpy(dtype=np.float64) for s in SPLITS}

display(split_summary)
print(f"{len(model_input):,} giao dịch; {len(FEATURE_COLUMNS)} đặc trưng.")


## 4. Hàm tính ba chỉ số

- **Silhouette:** cao hơn thì các cụm tách biệt và gọn hơn (khoảng cách Euclid).
- **Davies–Bouldin:** thấp hơn tốt hơn.
- **Calinski–Harabasz:** cao hơn tốt hơn khi so sánh cùng K; không dùng để chọn K.

Các chỉ số luôn tính trong không gian 11 đặc trưng `x_*`. Silhouette trên train dùng một
mẫu cố định (chọn một lần, dùng lại cho mọi K và seed) vì tính đủ rất chậm.

Nhãn lấy từ K-Means trên latent; dữ liệu tính chỉ số vẫn là `X[split]` gốc.

In [ ]:
rng = np.random.RandomState(EVAL_SEED)
silhouette_rows = {
    "train": rng.permutation(len(X["train"]))[:SILHOUETTE_TRAIN_SIZE],
    "validation": np.arange(len(X["validation"])),
    "test": np.arange(len(X["test"])),
}


def evaluate(split, labels):
    """Tính ba chỉ số cho một tập, trong không gian x_* gốc."""
    values = X[split]
    rows = silhouette_rows[split]
    scores = dict(n_eval=len(values), silhouette=None, davies_bouldin=None, calinski_harabasz=None)
    if not 2 <= np.unique(labels).size < len(values):
        return scores  # Các chỉ số không xác định nếu chỉ có một cụm.
    scores["davies_bouldin"] = float(davies_bouldin_score(values, labels))
    scores["calinski_harabasz"] = float(calinski_harabasz_score(values, labels))
    if 2 <= np.unique(labels[rows]).size < len(rows):
        scores["silhouette"] = float(silhouette_score(values[rows], labels[rows]))
    return scores


display(pd.DataFrame({
    "split": SPLITS,
    "n_eval": [len(X[s]) for s in SPLITS],
    "silhouette_n_eval": [len(silhouette_rows[s]) for s in SPLITS],
}))


## 5. Mạng Autoencoder và hàm suy luận

Mạng **11 → 16 → 8 → 4 → 8 → 16 → 11** dùng ReLU ở các tầng 16 và 8;
tầng latent và đầu ra tuyến tính. Đầu ra có thể âm vì các đặc trưng đã chuẩn hóa.

Autoencoder dùng tensor `float32`; các chỉ số phân cụm vẫn dùng ma trận gốc `float64`.
Hai hàm bên dưới tính MSE trên toàn bộ tập và mã hóa dữ liệu theo từng batch.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        n_in, h1, h2, latent, h3, h4, n_out = AE_PARAMS["architecture"]
        self.encoder = nn.Sequential(
            nn.Linear(n_in, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, h3), nn.ReLU(),
            nn.Linear(h3, h4), nn.ReLU(), nn.Linear(h4, n_out),
        )

    def forward(self, values):
        return self.decoder(self.encoder(values))


assert AE_PARAMS["architecture"][0] == AE_PARAMS["architecture"][-1] == len(FEATURE_COLUMNS)
T = {s: torch.from_numpy(np.ascontiguousarray(X[s], dtype=np.float32)) for s in SPLITS}


def reconstruction_mse(model, values):
    model.eval()
    squared_error = 0.0
    with torch.inference_mode():
        for batch in values.split(AE_PARAMS["inference_batch_size"]):
            residual = model(batch) - batch
            squared_error += residual.to(torch.float64).square().sum().item()
    return squared_error / values.numel()


def encode_values(model, values):
    model.eval()
    with torch.inference_mode():
        encoded = [model.encoder(batch).numpy() for batch in values.split(AE_PARAMS["inference_batch_size"])]
    return np.ascontiguousarray(np.concatenate(encoded), dtype=np.float32)


## 6. Huấn luyện và dừng sớm

Dùng Adam và MSE. Chỉ batch train đi qua `backward()` và `optimizer.step()`.
Cuối mỗi epoch, tính lại loss train và validation bằng cùng bộ trọng số.

Dừng khi MSE validation không giảm trong `patience` epoch liên tiếp.
Luôn khôi phục trọng số có MSE validation thấp nhất, kể cả khi chạy hết `max_epochs`.
Test không tham gia huấn luyện hoặc chọn checkpoint.

In [ ]:
def train_autoencoder(train_values, validation_values, seed):
    torch.manual_seed(seed)
    model = Autoencoder()
    optimizer = torch.optim.Adam(model.parameters(), lr=AE_PARAMS["lr"])
    criterion = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(train_values), batch_size=AE_PARAMS["batch_size"],
        shuffle=True, generator=torch.Generator().manual_seed(seed),
    )
    best_state = None
    best_mse = float("inf")
    best_epoch = 0
    stale_epochs = 0
    history = []
    started = perf_counter()

    for epoch in range(1, AE_PARAMS["max_epochs"] + 1):
        model.train()
        for (batch,) in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch), batch)
            if not torch.isfinite(loss):
                raise ValueError(f"Seed {seed}, epoch {epoch}: loss không hữu hạn.")
            loss.backward()
            optimizer.step()

        train_mse = reconstruction_mse(model, train_values)
        validation_mse = reconstruction_mse(model, validation_values)
        if not np.isfinite([train_mse, validation_mse]).all():
            raise ValueError(f"Seed {seed}, epoch {epoch}: MSE không hữu hạn.")
        if validation_mse < best_mse:
            best_mse = validation_mse
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        history.append({
            "seed": seed, "epoch": epoch,
            "train_mse": train_mse, "validation_mse": validation_mse,
        })
        if epoch == 1 or epoch % 20 == 0 or stale_epochs >= AE_PARAMS["patience"]:
            print(f"Seed {seed}, epoch {epoch}: train MSE={train_mse:.5f}, validation MSE={validation_mse:.5f}", flush=True)
        if stale_epochs >= AE_PARAMS["patience"]:
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    info = {
        "seed": int(seed), "epochs_run": epoch, "best_epoch": best_epoch,
        "stopped_early": epoch < AE_PARAMS["max_epochs"],
        "stop_reason": "patience" if stale_epochs >= AE_PARAMS["patience"] else "max_epochs",
        "best_validation_mse": float(best_mse),
        "fit_seconds": float(perf_counter() - started),
    }
    return model, pd.DataFrame(history), info


## 7. Huấn luyện một Autoencoder cho mỗi seed

Mỗi seed huấn luyện đúng một mạng. Giữ checkpoint tốt nhất và latent của ba tập
để dùng lại cho mọi K. `loss_history.csv` lưu loss từng epoch của tất cả seed.

In [ ]:
autoencoders, latent_by_seed, ae_training = {}, {}, {}
loss_frames = []
for seed in SEEDS:
    model, history, info = train_autoencoder(T["train"], T["validation"], seed)
    t0 = perf_counter()
    latent_by_seed[seed] = {s: encode_values(model, T[s]) for s in SPLITS}
    info["encode_seconds"] = perf_counter() - t0
    autoencoders[seed] = model
    ae_training[seed] = info
    loss_frames.append(history)
    print(f"Seed {seed}: giữ epoch {info['best_epoch']}/{info['epochs_run']}, MSE validation={info['best_validation_mse']:.6f}.", flush=True)

loss_history = pd.concat(loss_frames, ignore_index=True)
loss_history.to_csv(METHOD_DIR / "loss_history.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame(ae_training.values()).round(6))


## 8. Loss theo epoch

Mỗi ô là một seed: loss train màu xanh, validation nét đứt màu vàng,
đường dọc chỉ epoch được giữ. Các ô dùng cùng thang đo để dễ so sánh.

In [ ]:
fig, axes = plt.subplots((len(SEEDS) + 2) // 3, 3, figsize=(15, 4 * ((len(SEEDS) + 2) // 3)), squeeze=False, sharex=True, sharey=True)
for ax, seed in zip(axes.flat, SEEDS):
    history = loss_history.loc[loss_history["seed"].eq(seed)]
    ax.plot(history["epoch"], history["train_mse"], color="#2457A6", label="Train")
    ax.plot(history["epoch"], history["validation_mse"], color="#B27619", linestyle="--", label="Validation")
    ax.axvline(ae_training[seed]["best_epoch"], color="#555555", linestyle=":", label="Epoch được giữ")
    ax.set(title=f"Seed {seed} — giữ epoch {ae_training[seed]['best_epoch']}", xlabel="Epoch", ylabel="MSE / phần tử")
    ax.set_xlim(1, AE_PARAMS["max_epochs"])
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, fontsize=9)
for ax in axes.flat[len(SEEDS):]:
    ax.set_visible(False)
fig.suptitle("Autoencoder — loss tại cuối từng epoch trên train và validation")
fig.tight_layout()
save_figure(fig, "loss_by_epoch.png")


## 9. K-Means trên latent: K=3–10 × 5 seed

K-Means dùng `k-means++`, `n_init=10`, cùng cấu hình với notebook K-Means.
Mỗi seed dùng latent do Autoencoder của chính seed đó tạo ra. Chỉ fit K-Means trên
latent train; validation và test chỉ được gán nhãn bằng `predict`.

Ba chỉ số dùng `X[split]` **11 chiều gốc**. `recon_mse` chỉ có ở dòng validation
và không đổi theo K trong cùng seed; `inertia`, `bic`, `aic` để trống.

`fit_seconds` của mỗi cấu hình bằng thời gian huấn luyện Autoencoder + mã hóa ba tập
+ fit K-Means. Thời gian Autoencoder được dùng lại cho các K; **không cộng các dòng
của `k_sweep.csv` để tính tổng thời gian chạy**. Tổng thời gian fit thực tế được ghi
riêng trong `metrics.json`.

In [ ]:
rows, cluster_models, kmeans_fit_seconds = [], {}, {}
sweep_start = perf_counter()
for k in K_GRID:
    for seed in SEEDS:
        latent = latent_by_seed[seed]
        candidate = KMeans(n_clusters=k, random_state=seed, **KMEANS_PARAMS)
        t0 = perf_counter()
        candidate.fit(latent["train"])
        km_seconds = perf_counter() - t0
        cluster_models[(k, seed)] = candidate
        kmeans_fit_seconds[(k, seed)] = km_seconds
        info = ae_training[seed]
        fit_seconds = info["fit_seconds"] + info["encode_seconds"] + km_seconds
        for split in SPLITS:
            predicted = candidate.predict(latent[split])
            rows.append({
                "method": "autoencoder_kmeans", "k": k, "seed": seed, "split": split,
                **evaluate(split, predicted), "fit_seconds": fit_seconds,
                "inertia": None, "bic": None, "aic": None,
                "recon_mse": info["best_validation_mse"] if split == "validation" else None,
            })
    print(f"Đã xong K={k}: {len(SEEDS)} seed.", flush=True)

SWEEP_COLUMNS = [
    "method", "k", "seed", "split", "n_eval",
    "silhouette", "davies_bouldin", "calinski_harabasz", "fit_seconds",
    "inertia", "bic", "aic", "recon_mse",
]
k_sweep = pd.DataFrame(rows, columns=SWEEP_COLUMNS)
k_sweep.to_csv(METHOD_DIR / "k_sweep.csv", index=False, encoding="utf-8-sig")
print(f"Hoàn tất {len(cluster_models)} cặp (K, seed) trong {perf_counter() - sweep_start:.1f} giây sau bước Autoencoder.")
display(k_sweep.head(6).round(4))


## 10. Chọn K và seed bằng validation

Quy tắc (không đọc chỉ số test):

1. **K tốt nhất** là K có Silhouette validation trung bình (qua các seed) cao nhất.
2. **Seed tốt nhất** là seed có Silhouette validation cao nhất trong K đó.
3. Nếu bằng điểm thì chọn K nhỏ hơn / seed nhỏ hơn.

In [ ]:
validation_scores = k_sweep[k_sweep["split"] == "validation"]
if validation_scores["silhouette"].isna().any():
    raise ValueError("Có Silhouette validation không xác định; hãy kiểm tra nhãn cụm trước khi chọn K.")

selection_table = (
    validation_scores.groupby("k")["silhouette"]
    .agg(mean="mean", std="std")
    .fillna(0.0)
    .reset_index()
    .sort_values(["mean", "k"], ascending=[False, True])
    .reset_index(drop=True)
)
best_k = int(selection_table.loc[0, "k"])
best_validation_mean = float(selection_table.loc[0, "mean"])
best_validation_std = float(selection_table.loc[0, "std"])

best_seed = int(
    validation_scores[validation_scores["k"] == best_k]
    .sort_values(["silhouette", "seed"], ascending=[False, True])
    .iloc[0]["seed"]
)

display(selection_table.rename(columns={
    "mean": "validation_silhouette_mean", "std": "validation_silhouette_std",
}).round(6))
print(f"Chọn K={best_k}, seed={best_seed}.")
print(f"Silhouette validation trung bình: {best_validation_mean:.6f} ± {best_validation_std:.6f}.")
if best_k in (min(K_GRID), max(K_GRID)):
    print("Lưu ý: K được chọn nằm ở biên của lưới K; giá trị tốt hơn có thể nằm ngoài lưới.")


## 11. Ba chỉ số validation theo K

Đường biểu diễn trung bình qua các seed; dải mờ là ± một độ lệch chuẩn.
Đường dọc đánh dấu K được chọn bằng Silhouette. Calinski–Harabasz chỉ dùng
so sánh giữa các phương pháp ở cùng K, không dùng chọn K.

In [ ]:
def plot_mean_band(ax, data, column, color):
    """Vẽ trung bình theo K và dải ± 1 độ lệch chuẩn giữa các seed."""
    summary = data.groupby("k")[column].agg(["mean", "std"]).reindex(K_GRID)
    summary["std"] = summary["std"].fillna(0.0)
    ax.plot(K_GRID, summary["mean"], marker="o", color=color, linewidth=2)
    ax.fill_between(K_GRID, summary["mean"] - summary["std"], summary["mean"] + summary["std"],
                    color=color, alpha=0.15)
    ax.axvline(best_k, color="#555555", linestyle="--", linewidth=1)
    ax.set_xticks(K_GRID)
    ax.grid(axis="y", alpha=0.25)


metric_specs = [
    ("silhouette", "Silhouette (cao hơn tốt hơn)", "#2457A6"),
    ("davies_bouldin", "Davies–Bouldin (thấp hơn tốt hơn)", "#B27619"),
    ("calinski_harabasz", "Calinski–Harabasz (tham khảo)", "#8B4C70"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, title, color) in zip(axes, metric_specs):
    plot_mean_band(ax, validation_scores, metric, color)
    ax.set(title=title, xlabel="Số cụm K", ylabel="Giá trị chỉ số")
fig.suptitle(f"Autoencoder + K-Means trên validation — trung bình ± độ lệch chuẩn của {len(SEEDS)} seed")
fig.tight_layout()
save_figure(fig, "validation_metrics_by_k.png")


## 12. Mô hình cuối và kết quả test

Dùng lại Autoencoder và K-Means của cặp `(best_k, best_seed)` đã chọn;
không huấn luyện thêm mạng hoặc gộp validation vào train.
Bảng dưới tính cả ba chỉ số trên 11 đặc trưng gốc.

In [ ]:
final_autoencoder = autoencoders[best_seed]
final_model = cluster_models[(best_k, best_seed)]
final_latent = latent_by_seed[best_seed]
selected_training = ae_training[best_seed]
final_fit_seconds = (
    selected_training["fit_seconds"] + selected_training["encode_seconds"]
    + kmeans_fit_seconds[(best_k, best_seed)]
)
final_labels = {s: final_model.predict(final_latent[s]) for s in SPLITS}
final_scores = {s: evaluate(s, final_labels[s]) for s in SPLITS}

# Nhãn của cả ba tập để lưu và tra cứu theo cụm
labels = model_input[["transaction_id", "split"]].copy()
labels["cluster"] = -1
for s in SPLITS:
    labels.loc[split_masks[s], "cluster"] = final_labels[s]
assert labels["cluster"].between(0, best_k - 1).all()

final_score_table = pd.DataFrame.from_dict(final_scores, orient="index").reindex(SPLITS)
final_score_table.index.name = "split"
display(final_score_table.round(6))
display(Markdown(
    f"**Kết quả cuối trên test:** K={best_k}, seed={best_seed}; "
    f"Silhouette **{final_scores['test']['silhouette']:.4f}**, "
    f"Davies–Bouldin **{final_scores['test']['davies_bouldin']:.4f}**, "
    f"Calinski–Harabasz **{final_scores['test']['calinski_harabasz']:.2f}**."
))
print(f"Autoencoder giữ epoch {selected_training['best_epoch']}; MSE validation={selected_training['best_validation_mse']:.6f}.")


## 13. Mô tả cụm trên tập test

Bảng mô tả dùng nhãn cuối trên **test**, ghép theo `transaction_id` với
`features_unscaled.parquet` và `transaction_lookup.parquet`.
Số tiền tính bằng VND gốc; trung vị bỏ qua số tiền thiếu và ghi kèm số dòng thiếu.
Số giao dịch 7 ngày được khôi phục từ `log_sender_tx_count_7d` bằng `expm1`.

Các cờ «mới» và «đầu tiên» chỉ nói về lịch sử quan sát được. Số thứ tự cụm chỉ là mã định danh,
không thể hiện tốt hay xấu.

In [ ]:
profile_features = pd.read_parquet(
    PREPARED_DIR / "features_unscaled.parquet",
    columns=["transaction_id", "split", "is_new_receiver", "is_first_sender_tx", "log_sender_tx_count_7d"],
)
lookup = pd.read_parquet(
    PREPARED_DIR / "transaction_lookup.parquet",
    columns=["transaction_id", "transaction_datetime", "transactionamount"],
)

profile_data = (
    labels[labels["split"] == "test"]
    .merge(profile_features, on=["transaction_id", "split"], how="left", validate="one_to_one")
    .merge(lookup, on="transaction_id", how="left", validate="one_to_one")
)
assert len(profile_data) == len(X["test"])
assert profile_data["transaction_datetime"].notna().all(), "Không ghép được thông tin giao dịch"

profile_data["hour"] = profile_data["transaction_datetime"].dt.hour
profile_data["sender_tx_count_7d"] = np.rint(np.expm1(profile_data["log_sender_tx_count_7d"]))

cluster_profile = (
    profile_data.groupby("cluster")
    .agg(
        transaction_count=("transaction_id", "size"),
        median_amount_vnd=("transactionamount", "median"),
        modal_hour=("hour", lambda v: int(v.mode().iloc[0])),   # giờ phổ biến nhất (bằng nhau thì lấy giờ nhỏ hơn)
        is_new_receiver_rate=("is_new_receiver", "mean"),
        is_first_sender_tx_rate=("is_first_sender_tx", "mean"),
        median_sender_tx_count_7d=("sender_tx_count_7d", "median"),
        amount_missing_count=("transactionamount", lambda v: int(v.isna().sum())),
    )
    .reindex(range(best_k))
)
cluster_profile[["transaction_count", "amount_missing_count"]] = (
    cluster_profile[["transaction_count", "amount_missing_count"]].fillna(0).astype(int)
)
cluster_profile.insert(1, "transaction_share", cluster_profile["transaction_count"] / len(profile_data))
cluster_profile = cluster_profile.reset_index()

display(cluster_profile.style.format({
    "transaction_count": "{:,.0f}", "transaction_share": "{:.2%}",
    "median_amount_vnd": "{:,.0f}", "modal_hour": "{:.0f}",
    "is_new_receiver_rate": "{:.2%}", "is_first_sender_tx_rate": "{:.2%}",
    "median_sender_tx_count_7d": "{:.1f}", "amount_missing_count": "{:.0f}",
}, na_rep="—"))


## 14. Kích thước cụm

Biểu đồ thể hiện số giao dịch test trong từng cụm. Nhãn phía trên ghi cả số lượng
và tỷ lệ trên toàn bộ test, để nhận biết các cụm rất nhỏ hoặc chiếm ưu thế.

In [ ]:
cluster_colors = {c: plt.get_cmap("tab10")(c) for c in range(best_k)}

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(cluster_profile["cluster"], cluster_profile["transaction_count"],
       color=[cluster_colors[int(c)] for c in cluster_profile["cluster"]], width=0.7)
for row in cluster_profile.itertuples():
    ax.text(row.cluster, row.transaction_count,
            f"{row.transaction_count:,}\n({row.transaction_share:.1%})",
            ha="center", va="bottom", fontsize=10)
ax.set(title=f"Kích thước cụm Autoencoder + K-Means trên {len(X['test']):,} giao dịch test",
       xlabel="Mã cụm", ylabel="Số giao dịch",
       xticks=list(range(best_k)),
       ylim=(0, max(1, cluster_profile["transaction_count"].max()) * 1.22))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
save_figure(fig, "cluster_sizes_test.png")


## 15. Hình PCA hai chiều

PCA chỉ phục vụ hiển thị: fit PCA trên train rồi chiếu một mẫu tối đa 5.000 giao dịch test.
Màu là nhãn K-Means đã gán trong không gian **11 chiều**; không phân cụm lại trên hai tọa độ PCA.
Các điểm chồng nhau trong hình chưa đủ để kết luận các cụm trùng nhau trong toàn bộ không gian đặc trưng.
Nhãn lấy từ latent; PCA vẫn chiếu dữ liệu gốc để thống nhất với hai phương pháp còn lại.

In [ ]:
pca = PCA(n_components=2, svd_solver="full")
pca.fit(X["train"])

plot_rows = np.random.RandomState(EVAL_SEED).permutation(len(X["test"]))[:PCA_SAMPLE_SIZE]
projected = pca.transform(X["test"][plot_rows])
plot_labels = final_labels["test"][plot_rows]

fig, ax = plt.subplots(figsize=(10, 6))
for cluster in range(best_k):
    selected = plot_labels == cluster
    ax.scatter(projected[selected, 0], projected[selected, 1],
               color=cluster_colors[cluster], s=12, alpha=0.5, linewidths=0, label=f"Cụm {cluster}")
ax.set(title=f"PCA của {len(plot_rows):,} giao dịch test — màu theo cụm Autoencoder + K-Means",
       xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} phương sai train)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} phương sai train)")
ax.legend(title="Nhãn cụm", frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, "pca_test.png")
print(f"Hai thành phần PCA giữ {pca.explained_variance_ratio_.sum():.2%} phương sai trên train.")


## 16. Nhận xét và giới hạn

Các nhận xét sau lấy trực tiếp từ kết quả vừa chạy.

In [ ]:
largest = cluster_profile.loc[cluster_profile["transaction_count"].idxmax()]
most_active = cluster_profile.loc[cluster_profile["median_sender_tx_count_7d"].idxmax()]
highest_amount = cluster_profile.loc[cluster_profile["median_amount_vnd"].idxmax()]

display(Markdown(
    f"- K được chọn là **{best_k}** (seed {best_seed}).\n"
    f"- Cụm **{int(largest['cluster'])}** lớn nhất trên test, chiếm **{largest['transaction_share']:.1%}** giao dịch.\n"
    f"- Cụm **{int(most_active['cluster'])}** có trung vị số giao dịch 7 ngày của người gửi cao nhất: "
    f"**{most_active['median_sender_tx_count_7d']:.1f}**.\n"
    f"- Cụm **{int(highest_amount['cluster'])}** có số tiền trung vị cao nhất: "
    f"**{highest_amount['median_amount_vnd']:,.0f} VND**."
))


**Giới hạn.** MSE thấp cho biết mạng tái tạo đầu vào tốt, không bảo đảm cụm tách biệt tốt.
Nhãn được tạo trên latent nhưng đánh giá bằng khoảng cách ở không gian gốc; Silhouette thấp
cho thấy cụm còn chồng lấn theo thước đo này. Dữ liệu giả lập không có nhãn đúng,
nên các chỉ số không phải accuracy hay bằng chứng phát hiện gian lận.
Calinski–Harabasz chỉ so sánh giữa các phương pháp ở cùng K. Notebook 02 thực hiện so sánh ba phương pháp.

## 17. Lưu kết quả

Trong `output/autoencoder_kmeans/`: sáu file chung (`config.json`, `k_sweep.csv`,
`metrics.json`, `labels.parquet`, `cluster_profile.csv`, `model.joblib`),
`encoder.pt`, `loss_history.csv` và thư mục `figures/` chứa bốn hình.
Cấu hình chung ghi ở `output/config.json`.

`encoder.pt` chứa trọng số encoder và decoder, kiến trúc, thứ tự đặc trưng và epoch được chọn.
`model.joblib` chứa K-Means trên latent: **cần mã hóa dữ liệu qua encoder trước khi dự đoán**.
Ô cuối nạp lại hai thành phần để kiểm tra nhãn test.

In [ ]:
cluster_sizes = {
    s: {str(c): int(np.sum(final_labels[s] == c)) for c in range(best_k)} for s in SPLITS
}

# Cấu hình chung của lần chạy (ba phương pháp dùng cùng nội dung này)
run_config = {
    "run_id": RUN_ID,
    "k_grid": K_GRID,
    "seeds": SEEDS,
    "feature_columns": FEATURE_COLUMNS,
    "input_file": "output/prepared/model_input.parquet",
    "split_row_counts": {s: len(X[s]) for s in SPLITS},
    "metric_protocol": {
        "space": "prepared_x_features",
        "silhouette_train_sample_size": SILHOUETTE_TRAIN_SIZE,
        "silhouette_sample_seed": EVAL_SEED,
        "silhouette_validation_and_test": "all_rows",
        "davies_bouldin_and_calinski_harabasz": "all_rows_per_split",
    },
    "selection_rule": "K = highest mean validation silhouette over seeds; seed = best validation silhouette within K; test not used",
}
write_json(RUN_DIR / "config.json", run_config)

metrics = {
    "method": "autoencoder_kmeans", "run_id": RUN_ID,
    "best_k": best_k, "best_seed": best_seed,
    "best_validation_mean_silhouette": best_validation_mean,
    "best_validation_std_silhouette": best_validation_std,
    "metrics": final_scores,
    "non_empty_clusters": {split: int(np.unique(final_labels[split]).size) for split in SPLITS},
    "cluster_sizes": cluster_sizes,
    "final_fit_seconds": float(final_fit_seconds),
    "recon_mse_validation": selected_training["best_validation_mse"],
    "best_epoch": selected_training["best_epoch"],
    "epochs_run": selected_training["epochs_run"],
    "training_seconds_actual": float(sum(info["fit_seconds"] for info in ae_training.values()) + sum(kmeans_fit_seconds.values())),
    "encoding_seconds_actual": float(sum(info["encode_seconds"] for info in ae_training.values())),
    "cluster_profile_split": "test",
    "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
}
write_json(METHOD_DIR / "config.json", {
    **run_config,
    "method": "autoencoder_kmeans",
    "hyperparameters": {"autoencoder": AE_PARAMS, "kmeans": KMEANS_PARAMS},
    "ae_training_by_seed": list(ae_training.values()),
    "fit_seconds_definition": "AE training + encoding + KMeans fit; AE time reused across K, times repeated across splits",
    "recon_mse_definition": "Validation MSE per element at restored best AE checkpoint; only on validation rows",
    "final_model_policy": "reuse selected AE checkpoint and fitted KMeans from sweep; no refit",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
})
write_json(METHOD_DIR / "metrics.json", metrics)
labels.to_parquet(METHOD_DIR / "labels.parquet", index=False)
cluster_profile.to_csv(METHOD_DIR / "cluster_profile.csv", index=False, encoding="utf-8-sig")
joblib.dump(final_model, METHOD_DIR / "model.joblib")

checkpoint = {
    "format_version": 1,
    "architecture": AE_PARAMS["architecture"],
    "hidden_activation": "ReLU", "latent_activation": "linear", "output_activation": "linear",
    "feature_columns": FEATURE_COLUMNS,
    "seed": best_seed, "best_epoch": selected_training["best_epoch"],
    "best_validation_mse": selected_training["best_validation_mse"],
    "autoencoder_state_dict": {name: value.detach().cpu().clone() for name, value in final_autoencoder.state_dict().items()},
}
torch.save(checkpoint, METHOD_DIR / "encoder.pt")

# Nạp lại cả encoder và K-Means phải cho đúng nhãn test.
saved_checkpoint = torch.load(METHOD_DIR / "encoder.pt", map_location="cpu", weights_only=True)
saved_autoencoder = Autoencoder()
saved_autoencoder.load_state_dict(saved_checkpoint["autoencoder_state_dict"])
saved_model = joblib.load(METHOD_DIR / "model.joblib")
assert (saved_model.predict(encode_values(saved_autoencoder, T["test"])) == final_labels["test"]).all()

print("Đã lưu vào:", METHOD_DIR)
print("8 file kết quả và 4 biểu đồ trong figures/.")
